# 排序查找与集合操作

学习目标：排序并同步重排关联数据，在有序数组中查找位置，完成去重重建和集合筛选，并区分完整排序与 Top-k 选择。

前置知识：数组索引、布尔掩码、轴、切片、基本聚合。

运行环境：Python 3.12、NumPy 2.5；本章 descending 参数需要 NumPy 2.5 及以上。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的整数数据，后续单元沿用首次导入的 np。

## 1 排序数值

把任务耗时从短到长排列，可以直接使用 np.sort()。它默认升序排序并返回新数组，原输入不变。

下面 durations 保存四项任务的耗时，单位为秒。若确实要修改原数组，可调用数组自身的 sort() 方法；它原地排序，返回 None。

In [1]:
import numpy as np

durations = np.array([12, 5, 9, 5], dtype=np.int16)
ordered = np.sort(durations)

print(ordered, ordered.shape, ordered.dtype)  # [5 5 9 12] (4,) int16。
print(durations)  # [12 5 9 5]，原数组保持不变。

working = durations.copy()
working.sort()
print(working)  # [5 5 9 12]，原地排序后的工作副本。

[ 5  5  9 12] (4,) int16
[12  5  9  5]
[ 5  5  9 12]


## 2 argsort 与稳定性

### 2.1 按同一顺序重排关联数据

只排序耗时会丢失它与任务编号的对应关系。np.argsort() 返回排序所需的原位置索引，可以用同一组索引重排耗时与编号。

稳定排序（stable sort）保持相等排序值的原相对顺序。stable=True 明确要求稳定；默认不保证这一点。下面两项耗时同为 5 秒，排序后仍按原先出现顺序排列。

In [2]:
task_ids = np.array([101, 102, 103, 104], dtype=np.int16)
durations = np.array([12, 5, 9, 5], dtype=np.int16)
order = np.argsort(durations, stable=True)

print(order)  # [1 3 2 0]，每个数都是原数组的位置。
print(durations[order])  # [5 5 9 12]。
print(task_ids[order])  # [102 104 103 101]，相同耗时保留 102 在 104 之前。
print(order.shape, order.dtype)  # (4,)，本机索引 dtype 为 int64。

[1 3 2 0]
[ 5  5  9 12]
[102 104 103 101]
(4,) int64


### 2.2 稳定的降序排列

NumPy 2.5 提供 descending=True，可直接要求降序；与 stable=True 一起使用时，相等值仍保持原顺序。

把稳定升序的结果整体反转，会连相等值的相对顺序一起反转，不能用它替代稳定降序。下面按得分从高到低排列，编号仅用于辨认原顺序。

In [3]:
record_ids = np.array([101, 102, 103, 104], dtype=np.int16)
scores = np.array([80, 95, 80, 95], dtype=np.int16)
descending_order = np.argsort(scores, stable=True, descending=True)
reversed_order = np.argsort(scores, stable=True)[::-1]

print(scores[descending_order])  # [95 95 80 80]。
print(record_ids[descending_order])  # [102 104 101 103]，每组相等分数保持原顺序。
print(record_ids[reversed_order])  # [104 102 103 101]，每组相等分数的顺序被反转。

[95 95 80 80]
[102 104 101 103]
[104 102 103 101]


## 3 沿轴排序与取值

### 3.1 选择排序轴

下面 timings 的形状为 (2, 3)，两行表示两次实验，三列表示三个任务的耗时，单位为秒。

axis=1 在每行内排序，axis=0 在每列内排序；默认 axis=-1 表示最后一个轴。axis=None 先展平再整体排序。

沿某个轴排序会重排该轴的数值位置。如果还需要保留原任务或实验的对应关系，应同时取得索引。

In [4]:
timings = np.array([[9, 3, 6], [2, 8, 5]], dtype=np.int16)

print(np.sort(timings, axis=1))  # 两行为 [3 6 9]、[2 5 8]。
print(np.sort(timings, axis=0))  # 两行为 [2 3 5]、[9 8 6]。
print(np.sort(timings, axis=None))  # [2 3 5 6 8 9]，变为一维。
print(np.sort(timings, axis=1).shape)  # (2, 3)，沿轴排序保留原形状。

[[3 6 9]
 [2 5 8]]
[[2 3 5]
 [9 8 6]]
[2 3 5 6 8 9]
(2, 3)


### 3.2 take_along_axis 应用排序索引

二维 argsort() 返回的是每条轴内的索引。np.take_along_axis() 沿指定轴，按每组索引取回对应数值；这里索引数组与数据数组同形，axis 都设为 1。

沿用上面的 timings。order 的每一行给出相应实验内的任务排序位置，可以同时重排同形的任务编号表。不能把 timings[order] 当作按列取值，它会把 order 用在第一个轴上。

In [5]:
task_ids = np.array([[101, 102, 103], [101, 102, 103]], dtype=np.int16)
order = np.argsort(timings, axis=1, stable=True)
ordered_times = np.take_along_axis(timings, order, axis=1)
ordered_ids = np.take_along_axis(task_ids, order, axis=1)

print(order)  # 两行为 [1 2 0]、[0 2 1]。
print(ordered_times)  # 两行为 [3 6 9]、[2 5 8]。
print(ordered_ids)  # 两行为 [102 103 101]、[101 103 102]。
print(ordered_times.shape, ordered_times.dtype)  # (2, 3) int16。
print(np.array_equal(ordered_times, np.sort(timings, axis=1)))  # True。

[[1 2 0]
 [0 2 1]]
[[3 6 9]
 [2 5 8]]
[[102 103 101]
 [101 103 102]]
(2, 3) int16
True


## 4 searchsorted 查找插入位置

### 4.1 有序前提、左右边界与重复值

np.searchsorted() 查找“插入后仍保持升序”的位置，不会实际插入数据。默认要求一维输入已经升序排列。

side="left" 返回相等值之前的位置，side="right" 返回相等值之后的位置。因此，对同一个值，两位置之差就是它在有序数组中的出现次数。

插入位置可能等于数组长度，不能直接作为元素索引使用；找到插入位置也不代表该值已经存在。

In [6]:
ordered = np.array([10, 20, 20, 40], dtype=np.int16)
queries = np.array([5, 20, 25, 50], dtype=np.int16)
left = np.searchsorted(ordered, queries, side="left")
right = np.searchsorted(ordered, queries, side="right")

print(left)  # [0 1 3 4]，20 插在两个 20 之前。
print(right)  # [0 3 3 4]，20 插在两个 20 之后。
print(right - left)  # [0 2 0 0]，只有 20 已存在，出现两次。
print(ordered)  # [10 20 20 40]，没有发生实际插入。
print(ordered[left[2]])  # 40；查询 25 的插入位置不代表找到 25。
# 查询 50 返回长度 4，这是合法插入位置，但 ordered[4] 不存在。

[0 1 3 4]
[0 3 3 4]
[0 2 0 0]
[10 20 20 40]
40


### 4.2 未排序输入与 sorter

不能把未排序数组直接交给默认 searchsorted() 并期待正确位置。可以先排序，或通过 sorter 参数传入将原数组排成升序的索引。

使用 sorter 后，返回的仍是“排序后的数组”中的插入位置，不是原数组的下标。下面查找 25 应插入的位置，再把后继元素映射回原数据。

In [7]:
values = np.array([40, 10, 20], dtype=np.int16)
order = np.argsort(values)
position = np.searchsorted(values, 25, sorter=order)

print(values, values[order])  # 原顺序 [40 10 20]，有序顺序 [10 20 40]。
print(position)  # 2，指向排序后的第三个位置。
print(np.searchsorted(np.sort(values), 25))  # 同样为 2。
print(values[order[position]])  # 40，本例 position 小于数组长度，可以读取。

[40 10 20] [10 20 40]
2
2
40


## 5 unique 去重、计数与重建

np.unique() 默认返回升序排列的不重复值。return_counts=True 同时返回各值的出现次数；return_inverse=True 返回逆索引（inverse indices），指出原数组每个位置应从不重复值数组的哪里取值。

下面输入是一维类别编号。values[inverse] 可以恢复原值及顺序，单有计数不能恢复原先的排列。np.array_equal() 用于核对形状和值是否相同。

In [8]:
categories = np.array([30, 10, 30, 20, 10], dtype=np.int16)
values, inverse, counts = np.unique(
    categories, return_inverse=True, return_counts=True
)
restored = values[inverse]

print(values)  # [10 20 30]，按值排序，不是首次出现的顺序。
print(counts)  # [2 1 2]，分别对应 10、20、30。
print(inverse)  # [2 0 2 1 0]，指向 values 中的位置。
print(restored)  # [30 10 30 20 10]，恢复输入顺序。
print(restored.shape, restored.dtype)  # (5,) int16。
print(np.array_equal(restored, categories))  # True，形状和值均相同。
print(counts.sum() == categories.size)  # True，计数总和等于输入元素数。

[10 20 30]
[2 1 2]
[2 0 2 1 0]
[30 10 30 20 10]
(5,) int16
True
True


## 6 isin 筛选成员

np.isin() 逐元素判断某个值是否出现在候选集合中，返回与输入同形的布尔数组。用它筛选一维记录，可以保留原顺序和重复次数。

候选值使用列表或数组。若已有 Python set，先转换为列表；直接传 set 不会按预期展开其成员。invert=True 可以反向选择不在候选集合中的值。

In [9]:
record_ids = np.array([30, 10, 30, 20, 40], dtype=np.int16)
allowed = [10, 30]
mask = np.isin(record_ids, allowed)

print(mask, mask.shape, mask.dtype)  # [True True True False False] (5,) bool。
print(record_ids[mask])  # [30 10 30]，保留原顺序与重复值。
print(record_ids[np.isin(record_ids, allowed, invert=True)])  # [20 40]。
print(np.isin(record_ids, list({10, 30})))  # 与 mask 相同，先把 set 转为列表。

[ True  True  True False False] (5,) bool
[30 10 30]
[20 40]
[ True  True  True False False]


## 7 交集、并集与差集

需要比较两份编号名单时，可以使用以下集合函数。下表中 first、second 是两个输入数组；本节使用默认参数，结果为排序后去重的一维数组，不保留重复记录。

| 写法 | 中文名称／含义 |
| --- | --- |
| np.intersect1d(first, second) | 交集：两边都有的值 |
| np.union1d(first, second) | 并集：任一边出现过的值 |
| np.setdiff1d(first, second) | 差集：第一份中有、第二份中没有的值 |
| np.setxor1d(first, second) | 对称差集：只在其中一边出现的值 |

差集有方向，交换两个输入通常会得到不同结果。存在重复值时不要随意设置 assume_unique=True，它要求输入已经去重。

In [10]:
first = np.array([30, 10, 30, 20], dtype=np.int16)
second = np.array([20, 40, 20], dtype=np.int16)

print(np.intersect1d(first, second))  # [20]。
print(np.union1d(first, second))  # [10 20 30 40]。
print(np.setdiff1d(first, second))  # [10 30]，第一份独有。
print(np.setdiff1d(second, first))  # [40]，第二份独有。
print(np.setxor1d(first, second))  # [10 30 40]。

[20]
[10 20 30 40]
[10 30]
[40]
[10 30 40]


## 8 选学：多关键字排序

np.lexsort() 按多个关键字稳定排序，返回索引。传入序列的最后一个数组是主关键字，倒数第二个用于主关键字相同时比较，依此类推。

下面先按组号升序，再在同组内按耗时升序。两组关键字必须对应同一批记录、形状一致。

In [11]:
record_ids = np.array([101, 102, 103, 104, 105], dtype=np.int16)
groups = np.array([2, 1, 2, 1, 1], dtype=np.int16)
durations = np.array([5, 8, 3, 8, 4], dtype=np.int16)
order = np.lexsort((durations, groups))

print(order)  # [4 1 3 2 0]，groups 是主关键字。
print(groups[order])  # [1 1 1 2 2]。
print(durations[order])  # [4 8 8 3 5]，各组内部按耗时升序。
print(record_ids[order])  # [105 102 104 103 101]，完全相同关键字保留原顺序。

[4 1 3 2 0]
[1 1 1 2 2]
[4 8 8 3 5]
[105 102 104 103 101]


## 9 选学：分区与 Top-k

### 9.1 partition 只保证分区位置

如果只需要若干个较小值，可以用 np.partition() 分区。kth 是从 0 开始的位置索引；分区后，该位置的值与完整排序后的相同，其左侧不大于它、右侧不小于它。

两侧内部不保证有序，不能把分区当作完整排序。下面 kth=2 对应第三小的值，前面三个位置包含最小的三个值；原输入不变。

In [12]:
values = np.array([7, 2, 9, 4, 8, 1], dtype=np.int16)
partitioned = np.partition(values, kth=2)

print(partitioned)  # 位置 2 为 4，前三项为 1、2、4；其余位置不保证有序。
# 本次输出即使恰好全部有序，也不能据此认为 partition 保证完整排序。
print(np.sort(partitioned[:3]))  # [1 2 4]，只把选中的三个值排好。
print(partitioned[2] == np.sort(values)[2])  # True，分区位置与完整排序一致。
print(values)  # [7 2 9 4 8 1]，原输入不变。

[1 2 4 9 8 7]
[1 2 4]
True
[7 2 9 4 8 1]


### 9.2 argpartition 选取最大 k 项

Top-k 表示选取最高的 k 项，这里 k 是整数，满足 1 ≤ k ≤ 元素数量。np.argpartition() 返回分区索引，可以用它保留原记录的位置。

对于长度为 n 的一维数组，cut=n-k 是最大 k 项的起始位置。取分区索引的 cut: 部分得到候选成员，再对这些成员排序，才能形成有序的 Top-k 结果。下面输入没有并列值。

In [13]:
scores = np.array([7, 2, 9, 4, 8, 1], dtype=np.int16)
k = 3
cut = scores.size - k
candidate_indices = np.argpartition(scores, kth=cut)[cut:]
local_order = np.argsort(scores[candidate_indices], descending=True)
top_indices = candidate_indices[local_order]

print(scores[candidate_indices])  # 包含 7、8、9，但分区不保证它们的顺序。
print(top_indices)  # [2 4 0]，原数组中最高三项的位置。
print(scores[top_indices])  # [9 8 7]，对候选成员再次排序后得到降序。
expected = np.sort(scores, descending=True)[:k]
print(np.array_equal(scores[top_indices], expected))  # True，与完整排序的前 k 项一致。

[7 8 9]
[2 4 0]
[9 8 7]
True


### 9.3 边界并列值

argpartition() 不是稳定选择。多个元素与第 k 项同分时，不能要求它总选中最早出现的记录；事后只排序已选成员，也不能找回被排除的并列记录。

需要“恰好 k 项，同分按原顺序”时，可以采用稳定降序排序后取前 k 项。如果规则是“保留所有达到第 k 项分数的记录”，则按该分数筛选，数量可能超过 k。

In [14]:
scores = np.array([9, 8, 8, 8, 5], dtype=np.int16)
k = 2
cut = scores.size - k
candidate_indices = np.argpartition(scores, kth=cut)[cut:]
stable_indices = np.argsort(scores, descending=True, stable=True)[:k]
threshold = np.partition(scores, kth=cut)[cut]

print(candidate_indices, scores[candidate_indices])
# 两项分数为一个 9 和一个 8；被选中的是哪一个 8 不受稳定性保证。
print(stable_indices)  # [0 1]，稳定排序明确优先原先更早的并列项。
print(scores[scores >= threshold])  # [9 8 8 8]，保留全部边界并列值时有四项。
print(np.sort(scores[candidate_indices]))  # [8 9]，检查分数，不把某个并列索引当保证。

[1 0] [8 9]
[0 1]
[9 8 8 8]
[8 9]


## 本章小结

（1）sort 返回排序值，argsort 返回排序索引；稳定排序保持相等值的原相对顺序，关联数组使用同一组索引重排。

（2）沿轴索引配合 take_along_axis 取值。searchsorted 要求升序输入，返回插入位置，而不是存在性结论。

（3）unique 的计数说明频次，逆索引能恢复原顺序；isin 适合保留记录顺序和重复值，集合运算适合比较去重成员。

（4）分区不等于完整排序。Top-k 除了指定数量，还应明确并列值如何处理。

## 练习

（1）将下方记录按得分从高到低排序，分数相同的记录保持原顺序。打印排序后的编号和得分，说明为什么不能简单反转稳定升序的索引。

In [15]:
record_ids = np.array([201, 202, 203, 204], dtype=np.int16)
scores = np.array([70, 90, 70, 90], dtype=np.int16)

# 在此计算索引并同时重排编号和得分。
# 检查：编号顺序为 [202, 204, 201, 203]，分数为 [90, 90, 70, 70]。
# 在注释中说明稳定性要求和方法选择理由。

（2）先预测下面查询的左右插入位置及两者之差，再运行核对。哪些查询值存在？哪些插入位置不能直接用于数组取值？

In [16]:
ordered = np.array([2, 4, 4, 7], dtype=np.int16)
queries = np.array([1, 4, 5, 9], dtype=np.int16)

# 先在此记录预测，再解释重复值和数组末尾边界。
left = np.searchsorted(ordered, queries, side="left")
right = np.searchsorted(ordered, queries, side="right")
print(left)
print(right)
print(right - left)

[0 1 3 4]
[0 3 3 4]
[0 2 0 0]


（3）对下方类别数组去重，打印唯一值、计数和逆索引，再重建原数组。将重建结果与输入的形状、dtype 和数值分别核对，说明仅保存唯一值与计数为何不够。

In [17]:
categories = np.array([5, 2, 5, 3, 2, 5], dtype=np.int16)

# 在此取得 unique 的三个输出并重建。
# 检查：唯一值为 [2, 3, 5]，对应计数为 [2, 1, 3]。
# 重建结果应与输入同形、同 dtype、同顺序；可用 array_equal() 核对形状和值。

（4）有两种交付要求：① 保留下方记录中允许的编号，保持原顺序和重复次数；② 只列出出现过的允许编号，每个编号一次且升序。分别在 isin 筛选和 intersect1d 中选择方法，说明改变约束为何改变了选择。

In [18]:
record_ids = np.array([30, 10, 30, 20, 10], dtype=np.int16)
allowed = np.array([10, 30], dtype=np.int16)

# 需求①：选择方法并解释理由；检查结果为 [30, 10, 30, 10]。
# 需求②：选择方法并解释理由；检查结果为 [10, 30]。
# 两项操作都应保留原始 record_ids，不在输入上原地排序。

### 重点练习提示

对应第（4）题。先独立完成，再按需要查看提示。

（1）区分记录筛选与集合交集，检查重复次数和顺序是否属于交付内容。

（2）isin 先返回逐条记录的条件；intersect1d 直接返回共同的唯一值。

### 重点练习参考解析

对应第（4）题。

需求①用 isin(record_ids, allowed) 得到 [True, True, True, False, True]，再用该掩码筛选，结果为 [30, 10, 30, 10]，形状 (4,)。重复出现的记录和原先顺序都保留。

需求②用 intersect1d(record_ids, allowed)，结果为升序且去重的 [10, 30]，形状 (2,)。两份结果都是 int16，原数组不变。若用交集完成需求①，记录次数和顺序会丢失；这不是数值检查能够弥补的。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | 排序：[sort](https://numpy.org/doc/2.5/reference/generated/numpy.sort.html) 的 axis、stable、descending 与 Notes；[ndarray.sort](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.sort.html) 的原地排序；[argsort](https://numpy.org/doc/2.5/reference/generated/numpy.argsort.html) 的索引返回、稳定性及 descending（2.5 新增）；[take_along_axis](https://numpy.org/doc/2.5/reference/generated/numpy.take_along_axis.html) 的 axis、indices 与二维排序示例；[Indexing on ndarrays](https://numpy.org/doc/2.5/user/basics.indexing.html#integer-array-indexing) 的 Integer array indexing：单个索引数组作用于首轴。查找：[searchsorted](https://numpy.org/doc/2.5/reference/generated/numpy.searchsorted.html) 的升序条件、side、sorter、返回值与 sorter 示例。去重与筛选：[unique](https://numpy.org/doc/2.5/reference/generated/numpy.unique.html) 的 return_inverse、return_counts 与重建示例；[isin](https://numpy.org/doc/2.5/reference/generated/numpy.isin.html) 的同形布尔结果、invert 与 Notes 中 Python set 的处理；[array_equal](https://numpy.org/doc/2.5/reference/generated/numpy.array_equal.html) 的形状和值检查。集合：[intersect1d](https://numpy.org/doc/2.5/reference/generated/numpy.intersect1d.html) 的交集与 assume_unique 条件；[union1d](https://numpy.org/doc/2.5/reference/generated/numpy.union1d.html) 的排序去重并集；[setdiff1d](https://numpy.org/doc/2.5/reference/generated/numpy.setdiff1d.html) 的差集方向及默认排序；[setxor1d](https://numpy.org/doc/2.5/reference/generated/numpy.setxor1d.html) 的对称差集。选学：[lexsort](https://numpy.org/doc/2.5/reference/generated/numpy.lexsort.html) 的关键字优先级与稳定性；[partition](https://numpy.org/doc/2.5/reference/generated/numpy.partition.html) 的 kth、两侧顺序未定义；[argpartition](https://numpy.org/doc/2.5/reference/generated/numpy.argpartition.html) 的分区索引与 Notes 中不稳定选择。 |